# y_cut scan: clean flavour fraction of exclusive ee_kt + IFN jets

Which jet resolution gives the most events with a correct parton-level flavour assignment? For every y_cut the
event is clustered (ee_kt + IFN, hadronisation partons) into the exclusive jets at that y_cut, and classified with
the jets it actually has, whatever their number. Histogram `parton_ifn_ycut_vs_category` from
`jet-flv-test-histmaker.py` (`YCUT_SCAN`), one entry per event and y_cut, filled from a single clustering per event.
The jet flavours are the IFN flavours at the stage of the clustering where these jets exist
(`FlavHistory::flavour_at_step`), not after the full clustering.

* Clean: every jet carries at most one unit of flavour, and
  * H → qq̄: one jet has q content +1 and one −1, the others are gluon jets or resolved g → q′q̄′ pairs
    (q | q̄, q | q̄ | g, q | q̄ | g | g, q | q̄ | q′ | q̄′, ...);
  * H → gg: N_jets ≥ 2 + N_flavoured/2 (each gluon gives one flavourless jet or a resolved q′q̄′ pair).
* At parton level every input carries at most one unit of flavour, so for y_cut → 0 (every parton its own jet)
  every event is trivially "clean". The figure of merit is therefore the fraction of events that are clean **with
  at most `N_MAX` jets**; clean events with more jets count as over-resolved.
* Top pad: composition vs. log10(y_cut), fractions of all events (stacked). Bottom pad: clean fraction with
  N_jets ≤ 3 and ≤ 4, the maximum marked, and the exclusive 2-jet clustering as reference (dashed). The cut value
  is the lower edge of its bin.
* Output: `plots/parton_ifn/ycut_scan/<sample>.pdf`, `all_samples_ycut_scan.pdf` (one page per sample) and
  `summary_ycut_scan.pdf` (all samples, N_jets ≤ `N_MAX`).

In [1]:
import os
import glob
import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)

# pixel text sizes (same in both pads of two-pad plots) and legends placed in free space
from plot_helpers import (pixel_axes, make_legend, place_legend, draw_legend_at, headroom, LEFT_MARGIN, X_TITLE_OFFSET,
                          Y_TITLE_OFFSET, BOTTOM_MARGIN_PX)

INPUT_DIR = os.path.join("..", "..", "..", "outputs", "jet-test")
OUT_DIR = os.path.join(INPUT_DIR, "plots", "parton_ifn", "ycut_scan")
HIST = "parton_ifn_ycut_vs_category"
XRANGE = (-5., 0.)  # log10(y_cut) shown
YRANGE = (0.9, 1.0)  # clean fraction shown in the bottom pad and the summary
N_MAX = 4  # summary plot: clean with at most N_MAX jets
# category = min(N_jets, 5) - 1 (clean) or 5 + min(N_jets, 5) - 1 (unclean), see the histmaker
STACK = [  # (label, categories, colour)
    ("clean, 2 jets", [1], ROOT.kBlue + 1),
    ("clean, 3 jets", [2], ROOT.kAzure + 7),
    ("clean, 4 jets", [3], ROOT.kCyan - 7),
    ("clean, #geq 5 jets", [4], ROOT.kGray),
    ("not clean", [5, 6, 7, 8, 9], ROOT.kRed + 1),
]

os.makedirs(OUT_DIR, exist_ok=True)
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.root")))
files

['../../../outputs/jet-test/wzp6_ee_nunuH_Hbb_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hcc_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hdd_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hgg_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hss_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Huu_ecm240.root']

In [2]:
def fractions(h2, cats):
    """Fraction of all events in the given categories, vs. log10(y_cut) (TH1)."""
    n = h2.Integral(1, 1, 0, h2.GetNbinsY() + 1)  # every event fills each column once
    h = None
    for cat in cats:
        p = h2.ProjectionX(f"{h2.GetName()}_{cat}_{len(cats)}", cat + 1, cat + 1)
        if h is None:
            h = p
        else:
            h.Add(p)
    h.SetDirectory(0)
    h.Scale(1. / n)
    return h


def clean_upto(h2, n_max):
    return fractions(h2, list(range(1, min(n_max, 5))))  # clean with 2 .. n_max jets


def best(h):
    """(log10 y_cut, fraction) of the maximum within XRANGE; y_cut is the lower bin edge."""
    i_lo, i_hi = h.GetXaxis().FindFixBin(XRANGE[0]), h.GetXaxis().FindFixBin(XRANGE[1]) - 1
    i = max(range(i_lo, i_hi + 1), key=h.GetBinContent)
    return h.GetXaxis().GetBinLowEdge(i), h.GetBinContent(i)


def two_jet_reference(tfile):
    """Clean fraction of the exclusive 2-jet ee_kt + IFN clustering."""
    h_all, h_c = tfile.Get("parton_ifn_log10_y23"), tfile.Get("parton_ifn_log10_y23_clean")
    return h_c.Integral(0, h_c.GetNbinsX() + 1) / h_all.Integral(0, h_all.GetNbinsX() + 1)

In [3]:
def draw(tfile, sample):
    h2 = tfile.Get(HIST)
    h2.SetName(f"{sample}_{HIST}")
    ref = two_jet_reference(tfile)

    c = ROOT.TCanvas(f"ycut_{sample}", "", 700, 800)
    top = ROOT.TPad("top", "", 0, 0.42, 1, 1)
    bot = ROOT.TPad("bot", "", 0, 0, 1, 0.42)
    top.SetBottomMargin(0.02)
    top.SetTopMargin(0.08)
    bot.SetTopMargin(0.03)
    bot.SetBottomMargin(BOTTOM_MARGIN_PX / (c.GetWh() * 0.42))
    for p in (top, bot):
        p.SetGridx()
        p.SetLeftMargin(LEFT_MARGIN)
        p.Draw()

    top.cd()
    stack = ROOT.THStack(f"stack_{sample}", f"{sample}, parton level, ee_kt + IFN;;fraction of all events")
    hists = []
    for label, cats, col in STACK:
        h = fractions(h2, cats)
        h.SetFillColor(col)
        h.SetLineColor(col)
        stack.Add(h, "hist")
        hists.append((h, label))
    stack.Draw()
    stack.GetXaxis().SetLimits(*XRANGE)
    pixel_axes(stack.GetXaxis(), stack.GetYaxis())
    stack.GetXaxis().SetLabelSize(0)
    stack.GetYaxis().SetTitleOffset(Y_TITLE_OFFSET)
    stack.SetMaximum(1.)
    top.Modified()
    # the stack fills the whole frame, so the legend goes into headroom above 1
    leg, w, h = make_legend([(h, label, "f") for h, label in reversed(hists)], ncols=2)
    place_legend(top, leg, w, h, [stack], grow=headroom(stack, 0., 1.), max_steps=20)

    bot.cd()
    curves = [(clean_upto(h2, n), n, col) for n, col in ((3, ROOT.kAzure + 7), (4, ROOT.kBlue + 1))]
    frame = bot.DrawFrame(XRANGE[0], YRANGE[0], XRANGE[1], YRANGE[1])
    frame.SetTitle(";log_{10}(y_{cut});clean fraction")
    pixel_axes(frame.GetXaxis(), frame.GetYaxis())
    frame.GetXaxis().SetTitleOffset(X_TITLE_OFFSET)
    frame.GetYaxis().SetTitleOffset(Y_TITLE_OFFSET)
    frame.GetYaxis().SetNdivisions(505)
    line = ROOT.TLine(XRANGE[0], ref, XRANGE[1], ref)
    line.SetLineStyle(2)
    line.SetLineWidth(2)
    line.Draw()
    keep = [line]
    entries = []
    for h, n, col in curves:
        h.SetLineColor(col)
        h.SetLineWidth(2)
        h.GetXaxis().SetRangeUser(*XRANGE)
        h.Draw("hist same")
        x, y = best(h)
        m = ROOT.TMarker(x + 0.5 * h.GetBinWidth(1), y, 20)
        m.SetMarkerColor(col)
        m.Draw()
        keep.append(m)
        entries.append((h, f"N_{{jets}} #leq {n}: max {y:.1%}", "l"))
    entries.append((line, f"excl. 2 jets: {ref:.1%}", "l"))
    leg2, w, h = make_legend(entries)
    place_legend(bot, leg2, w, h, [h for h, _, _ in curves] + keep, grow=headroom(frame, *YRANGE))

    c.SaveAs(os.path.join(OUT_DIR, f"{sample}.pdf"))
    c.Draw()
    c._keep = (top, bot, stack, hists, leg, frame, curves, leg2, keep)
    return c, {n: (h, best(h)) for h, n, _ in curves}, ref

In [4]:
canvases, results = [], {}
for fname in files:
    sample = os.path.basename(fname)[:-5]
    tfile = ROOT.TFile.Open(fname)
    c, curves, ref = draw(tfile, sample)
    canvases.append(c)
    results[sample] = (curves, ref)
    tfile.Close()

combined = os.path.join(OUT_DIR, "all_samples_ycut_scan.pdf")
canvases[0].Print(combined + "[")
for c in canvases:
    c.Print(combined, f"Title:{c.GetName()}")
canvases[-1].Print(combined + "]")

print(f"{'sample':28s} {'excl. 2 jets':>12s}   " + "   ".join(f"N<={n}: max (log10 ycut)" for n in (3, 4)))
for sample, (curves, ref) in results.items():
    row = "   ".join(f"{curves[n][1][1]:.3f} ({curves[n][1][0]:+.1f})      " for n in (3, 4))
    print(f"{sample:28s} {ref:12.3f}   {row}")

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/wzp6_ee_nunuH_Hgg_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/wzp6_ee_nunuH_Hss_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/wzp6_ee_nunuH_Huu_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf has been created
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf


Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/all_samples_ycut_scan.pdf has been closed


sample                       excl. 2 jets   N<=3: max (log10 ycut)   N<=4: max (log10 ycut)
wzp6_ee_nunuH_Hbb_ecm240            0.955   0.976 (-1.4)         0.986 (-2.0)      
wzp6_ee_nunuH_Hcc_ecm240            0.949   0.970 (-1.4)         0.980 (-2.0)      
wzp6_ee_nunuH_Hdd_ecm240            0.946   0.967 (-1.5)         0.977 (-2.0)      
wzp6_ee_nunuH_Hgg_ecm240            0.947   0.964 (-1.3)         0.980 (-1.8)      
wzp6_ee_nunuH_Hss_ecm240            0.946   0.967 (-1.5)         0.977 (-2.0)      
wzp6_ee_nunuH_Huu_ecm240            0.944   0.966 (-1.5)         0.976 (-2.0)      


In [5]:
# summary: clean fraction with at most N_MAX jets for all samples
colours = [ROOT.kBlue + 1, ROOT.kRed + 1, ROOT.kGreen + 2, ROOT.kOrange + 7, ROOT.kMagenta + 1, ROOT.kBlack]
cs = ROOT.TCanvas("ycut_summary", "", 700, 650)
cs.SetGridx()
cs.SetGridy()
cs.SetLeftMargin(LEFT_MARGIN)
cs.SetBottomMargin(BOTTOM_MARGIN_PX / cs.GetWh())
curves = [(s, r[0][N_MAX][0]) for s, r in results.items()]
frame = cs.DrawFrame(XRANGE[0], YRANGE[0], XRANGE[1], YRANGE[1])
frame.SetTitle(f"parton level, ee_kt + IFN;log_{{10}}(y_{{cut}});clean fraction (N_{{jets}} #leq {N_MAX})")
pixel_axes(frame.GetXaxis(), frame.GetYaxis())
frame.GetXaxis().SetTitleOffset(X_TITLE_OFFSET)
frame.GetYaxis().SetTitleOffset(Y_TITLE_OFFSET)
hists, entries = [], []
for (sample, h), col in zip(curves, colours):
    h = h.Clone(f"summary_{sample}")
    h.SetLineColor(col)
    h.SetLineWidth(2)
    h.GetXaxis().SetRangeUser(*XRANGE)
    h.Draw("hist same")
    x, y = results[sample][0][N_MAX][1]
    entries.append((h, f"H#rightarrow{sample.split('_H')[-1].split('_')[0]}: {y:.1%} at {x:.1f}", "l"))
    hists.append(h)
leg, w, h = make_legend(entries)
placed = place_legend(cs, leg, w, h, hists)
if placed:
    leg = placed
else:
    # no room in the frame: two columns below the plot
    leg, w, h = make_legend(entries, ncols=2)
    cs.SetCanvasSize(700, int(650 + h))
    cs.SetBottomMargin((BOTTOM_MARGIN_PX + h) / cs.GetWh())
    leg = draw_legend_at(cs, leg, 0.5 - 0.5 * w / cs.GetWw(), 5. / cs.GetWh(), 0.5 + 0.5 * w / cs.GetWw(),
                         (5. + h) / cs.GetWh())
cs._keep = hists + [leg]
cs.SaveAs(os.path.join(OUT_DIR, "summary_ycut_scan.pdf"))
cs.Draw()

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/ycut_scan/summary_ycut_scan.pdf has been created
